# 前処理① データ整形コードやimportなど

In [ ]:
!pip install osmnx
!pip install optuna

from google.colab import drive
import pandas as pd
import numpy as np
import os
import lightgbm as lgb
from lightgbm import LGBMRegressor, LGBMClassifier
from copy import deepcopy
import osmnx as ox
import geopandas as gpd
import fiona
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.tree import DecisionTreeRegressor
import joblib
import matplotlib.pyplot as plt
import optuna

In [ ]:
os.chdir("/content/drive/MyDrive/ml_projects/mlit_geospatial_challenge_2")

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# df = pd.read_csv("/content/drive/MyDrive/ml_projects/mlit_geospatial_challenge_2/data/raw/train.csv")

# stations_data = gpd.read_file("/content/drive/MyDrive/ml_projects/mlit_geospatial_challenge_2/data/raw/stations_japan_osm.gpkg", layer="stations")
# landprice_data = gpd.read_file("/content/drive/MyDrive/ml_projects/mlit_geospatial_challenge_2/data/raw/L01-25.geojson")

df = pd.read_csv("data/raw/train.csv")

stations_data = gpd.read_file("data/raw/stations_japan_osm.gpkg", layer="stations")
landprice_data = gpd.read_file("data/raw/L01-25.geojson")

TARGET = 'money_room'

In [ ]:
def drop_after_feature_engineering(df):
  # df = df.copy(deep = True)
  drop_cols = [
    "target_ym",
    # "money_room",
    "building_id",
    # "building_status",
    "building_create_date",
    "building_modify_date",
    # "building_type",
    "building_name",
    "building_name_ruby",
    "homes_building_name",
    "homes_building_name_ruby",
    # "unit_count",
    "full_address",
    # "lon",
    # "lat",
    # "building_structure",
    # "total_floor_area",
    # "building_area",
    # "floor_count",           # bin化
    # "basement_floor_count",
    "year_built",              # bin化
    "building_land_area",      # bin化
    "land_area_all",
    "unit_area_min",
    "unit_area_max",
    "building_land_chimoku", # フラグ化した別カラムを作成
    # "land_youto",
    # "land_toshi",
    "land_chisei",
    "land_area_kind",
    "land_setback_flg",
    "land_setback",
    "land_kenpei",
    "land_youseki",
    "land_road_cond",
    "land_seigen",
    "building_area_kind",
    "management_form",
    "management_association_flg",
    "reform_exterior",
    "reform_exterior_other",
    "reform_exterior_date",
    "reform_common_area",
    "reform_common_area_date",
    "building_tag_id", # one-hot化
    "unit_id",
    "unit_name",
    "name_ruby",
    "room_floor",
    "balcony_area",
    "dwelling_unit_window_angle",
    "room_count",
    # "unit_area",
    "floor_plan_code", # 109madori_kind_allと内容がかぶるため、一旦削除
    "reform_date",
    "reform_place",
    "reform_place_other",
    "reform_wet_area",
    "reform_wet_area_other",
    "reform_wet_area_date",
    "reform_interior",
    "reform_interior_other",
    "reform_interior_date",
    "reform_etc",
    "renovation_date",
    "renovation_etc",
    "unit_tag_id", # onehot化
    "bukken_id",
    "snapshot_create_date",
    "new_date",
    "snapshot_modify_date",
    "timelimit_date",
    # "bukken_type",
    "flg_investment",
    "empty_number",
    "empty_contents",
    "post1",       #郵便番号
    "post2",       #郵便番号
    # "addr1_1",
    "addr1_2",     #addr1_1とaddr1_2を合わせた別カラムを作成
    "addr2_name",
    "addr3_name",
    "nl",
    "el",
    "rosen_name1",
    "eki_name1",
    "bus_stop1",
    "bus_time1",
    "walk_distance1",
    "rosen_name2",
    "eki_name2",
    "bus_stop2",
    "bus_time2",
    "walk_distance2",
    "traffic_other",
    "traffic_car",
    "snapshot_land_area",
    "snapshot_land_shidou",
    "land_shidou_a",
    "land_shidou_b",
    "land_mochibun_a",
    "land_mochibun_b",
    "house_area",
    # "flg_new",
    "house_kanrinin",
    # "room_kaisuu",         #bin化
    "snapshot_window_angle", #one-hotエンコーディングして、その後に削除
    "madori_number_all",     #bin化
    "madori_kind_all",       #bin化
    # "money_kyoueki",
    "money_kyoueki_tax",
    "money_rimawari_now",
    "money_shuuzen",
    "money_shuuzenkikin",
    "money_sonota_str1",
    "money_sonota1",
    "money_sonota_str2",
    "money_sonota2",
    "money_sonota_str3",
    "money_sonota3",
    "parking_money",
    "parking_money_tax",
    "parking_kubun",
    "parking_distance",
    "parking_number",
    "parking_memo",
    "genkyo_code",
    "usable_status",
    "usable_date",
    "school_ele_name",
    "school_ele_distance",
    "school_ele_code",
    "school_jun_name",
    "school_jun_distance",
    "school_jun_code",
    "convenience_distance",
    "super_distance",
    "hospital_distance",
    "park_distance",
    "drugstore_distance",
    "bank_distance",
    "shopping_street_distance",
    "est_other_name",
    "est_other_distance",
    "statuses", # onehot化
    "parking_keiyaku",
    "money_hoshou_company",
    "free_rent_duration",
    "free_rent_gen_timing"
  ]

  df = df.drop(columns = drop_cols)
  return df

In [ ]:
#4 building_statusの1と9の二値を1と0の二値に変換
def preprocess_building_status(df):
  df["building_status"] = (df["building_status"] == 1).astype(int)
  return df


#7 building_typeを1,3,4だけ残してあとはその他として処理
def preprocess_building_type(df):
  df['building_type'] = df['building_type'].where(df['building_type'].isin([1, 3, 4]), 0)
  return df


#12 unit_countの欠損値をbuilding_typeの平均ごとに穴埋め
def preprocess_unit_count(df):
  df['unit_count'] = df.groupby('building_type')['unit_count'].transform(
    lambda x: x.fillna(x.median()) # building_type=0 はその他をまとめた中央値で補完
  )
  return df


#14 lonをメッシュ化したカラムを作成
mesh_size=0.01
def preprocess_lon(df):
  df["lon_mesh"] = (df["lon"] / mesh_size).astype(int) #あとでcategory化
  return df


#15 latをメッシュ化したカラムを作成
def preprocess_lat(df):
  df["lat_mesh"] = (df["lat"] / mesh_size).astype(int) #あとでcategory化
  return df


#16 building_structureのNaNを4で埋める(4……RC / 最頻の4の記入が省かれている可能性が高い為)
def preprocess_building_structure(df):
  df['building_structure'] = df['building_structure'].fillna(4.0)
  return df


#17 total_floor_areaの-1とNaNを中央値で埋め、対数変換した別カラムを作成する
def preprocess_total_floor_area(df):
  df["total_floor_area"] = df["total_floor_area"].replace(-1, np.nan)
  df["total_floor_area"] = df["total_floor_area"].fillna(df["total_floor_area"].median())
  df["log_total_floor_area"] = np.log1p(df["total_floor_area"])
  return df


#18 building_areaの-1とNaNを中央値で埋め、対数変換した別カラムを作成する
def preprocess_building_area(df):
  df["building_area"] = df["building_area"].replace(-1, np.nan)
  df["building_area"] = df["building_area"].fillna(df["building_area"].median())
  df["log_building_area"] = np.log1p(df["building_area"])
  return df


#19 floor_countの0と外れ値をNaNに変換。階層ごとにbin化した別カラムを作成する
def preprocess_floor_count(df):
  df.loc[df["floor_count"] == 0, "floor_count"] = np.nan
  df.loc[df["floor_count"] > 100, "floor_count"] = np.nan
  df.loc[df["floor_count"] < -10, "floor_count"] = np.nan
  bins = [-np.inf, 0, 1, 2, 3, 6, 10, 15, 20, 30, 40, 50, 60, np.inf]
  labels = [
    "B",
    "1", "2", "3", "4-6", "7-10",
    "11-15", "16-20", "21-30", "31-40",
    "41-50", "51-60", "60+"
  ]
  df["floor_count_bin"] = pd.cut(
    df["floor_count"],
    bins=bins,
    labels=labels
  )
  return df


#20 basement_floor_countを0と1の2値にする
def preprocess_basement_floor_count(df):
  df["basement_floor_count"] = df["basement_floor_count"].fillna(0)
  df["basement_floor_count"] = df["basement_floor_count"].apply(lambda x: 1 if x > 0 else 0)
  return df


#21 year_builtの外れ値をNaNにしてbin化したカラムの追加
def preprocess_year_built(df):
  # 外れ値を NaN
  year = df['year_built'].where(
    df['year_built'].between(190001, 204012),
    other=pd.NA
  )
  year = (year // 100).astype('Int64')
  bins = [0, 1950, 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2041]
  labels = [
    '〜1950', '1950–1969', '1970–1979', '1980–1989',
    '1990–1999', '2000–2009', '2010–2014',
    '2015–2019', '2020–2040'
  ]
  df['year_built_bin'] = pd.cut(
    year,
    bins=bins,
    labels=labels,
    right=True,
    include_lowest=True
  )
  return df


#22 building_land_areaの外れ値をNaNにしてbin化した別カラムの作成
def preprocess_building_land_area(df):
  df = df.copy()
  area = df['building_land_area'].where(
    df['building_land_area'] > 0,
    other=pd.NA
  )
  bins = [
    0, 20, 40, 60, 80, 100,
    200, 250, 300, 350, 400,
    450, 500, 600, 700, 800,
    900, 1000, float('inf')
  ]
  labels = [
    '0–20', '21–40', '41–60', '61–80', '81–100',
    '101–200', '201–250', '251–300', '301–350',
    '351–400', '401–450', '451–500', '501–600',
    '601–700', '701–800', '801–900', '901–1000', '1000+'
  ]
  df['building_land_area_bin'] = pd.cut(
    area,
    bins=bins,
    labels=labels,
    include_lowest=True
  )
  return df



#26 building_land_chimokuを1 or NaNでフラグ化した別カラムを作成
def preprocess_building_land_chimoku(df):
  # 宅地 or 住宅系フラグ
  # 1 or NaN → 1、それ以外 → 0
  df["is_takuchi"] = (
    (df["building_land_chimoku"] == 1) |
    (df["building_land_chimoku"].isna())
  )
  return df


#27 land_youtoをマッピング(1,10を合わせて低層住居専用地域に)
def preprocess_land_youto(df):
  youto_map = {
    1: "低層住居専用地域",
    10: "低層住居専用地域",
    2: "第二種中高層住居専用地域",
    3: "第二種住居地域",
    4: "近隣商業地域",
    5: "商業地域",
    6: "準工業地域",
    7: "工業地域",
    8: "工業専用地域",
    9: "その他",
    11: "第一種中高層住居専用地域",
    12: "第一種住居地域",
    13: "準住居地域",
    14: "田園住居地域",
    99: "未指定"
  }
  df["land_youto"] = df["land_youto"].map(youto_map).fillna("未指定")
  return df


#28 land_toshiをマッピング
def preprocess_land_toshi(df):
  toshi_map = {
    1: "市街化区域",
    2: "市街化調整区域" ,
    3: "非線引区域" ,
    4: "都市計画区域外"
  }
  df['land_toshi'] = df['land_toshi'].map(toshi_map) # マッピングしてカテゴリ化
  df['land_toshi'] = df['land_toshi'].fillna('unknown') # NaN を文字列 "unknown" にする
  return df


#45 building_tag_id ,67 unit_tag_id ,145 statuses の使えそうなものをone-hot化。ガス,コンロを3値で保持
def preprocess_building_tag_id(df):
  onehot_tags = [
    "113501",
    "210302","210401",
    "220401","220801",
    "223201","223401","223501","223601",
    "230202","230203","230401","230501","230601","230701",
    "233101",
    "240201",
    "250101","250201","250301",
    "253401","253701",
    "260101","260201","260301","260501","260503",
    "290201","290301","290501","290601",
    "293101","294201",
    "310101","310201","310401","310501",
    "313101",
    "320101","320201","320401","320501","321101",
    "330101","330501",
    "331001",
    "334001","334101","334201",
    "340101","340102","340201","340301","340401"
  ]
  tags_all = (
    df["building_tag_id"].fillna("")
    + "/"
    + df["unit_tag_id"].fillna("")
    + "/"
    + df["statuses"].fillna("")
  )
  #one-hot化
  for tag in onehot_tags:
    df[f"tag_{tag}"] = tags_all.str.contains(tag).astype(int)
  #ガス種別
  df["gas_type"] = 0
  df.loc[tags_all.str.contains("210201"), "gas_type"] = 1  # 都市ガス
  df.loc[tags_all.str.contains("210202"), "gas_type"] = 2  # プロパン
  #コンロ種別
  df["stove_type"] = 0
  df.loc[tags_all.str.contains("230101"), "stove_type"] = 1  # ガス
  df.loc[tags_all.str.contains("230102"), "stove_type"] = 2  # 電気
  df.loc[tags_all.str.contains("230103"), "stove_type"] = 3  # IH
  return df


# #49 room_floorをbin化。NaNはNaNとして保持
# def preprocess_room_floor(df):
#   bins = [-np.inf, 0, 1, 2, 3, 6, 10, 15, 20, 30, 40, 50, 60, np.inf]
#   labels = ["0以下", "1", "2", "3", "4-6", "7-10", "11-15", "16-20",
#             "21-30", "31-40", "41-50", "51-60", "60+"]
#   df["room_floor"] = pd.cut(
#     df["room_floor"],
#     bins=bins,
#     labels=labels
#   )
#   return df


#52 room_countをbin化
def preprocess_room_count(df):
  df.loc[df['room_count'] < 1, 'room_count'] = np.nan
  df['room_count'] = pd.cut(
      df['room_count'],
      bins=[0, 1, 2, 3, 4, 5, 6, 8, 10, np.inf],
      labels=['1', '2', '3', '4', '5', '6', '7-8', '9-10', '11+']
  )
  return df


#53 NaNを中央値で埋め、対数変換
def preprocess_unit_area(df):
  df['unit_area'] = np.log1p(df['unit_area'].fillna(df['unit_area'].median()))
  return df


#54 floor_plan_code間取りのみ取り出し → 109madori_kind_allと内容がかぶるため、一旦コメントアウト
# def preprocess_floor_plan_code(df):
#   plan_map = {10:'R', 20:'K', 30:'DK', 40:'LK', 50:'LDK'}
#   df['floor_plan_code'] = (
#     df['floor_plan_code'].astype('Int64') % 100
#   ).map(plan_map)
#   return df


#73 bukken_typeの1202を0、1302を1のフラグ特徴量に
def preprocess_bukken_type(df):
  df["bukken_type"] = (df["bukken_type"] == 1302).astype(int)
  return df


#79,80 addr1_1とaddr1_2を合わせた別カラムを作成
def preprocess_addr1(df):
  df["pref_city"] = (df["addr1_1"].astype(str) + "_" + df["addr1_2"].astype(str))
  return df


#104 flg_newの1をそのまま、それ以外を0に
def preprocess_flg_new(df):
  df["flg_new"] = (df["flg_new"] == 1).astype(int)
  return df


#106 room_kaisuuの外れ値をNaNにそれ以外をbin化
def preprocess_room_kaisuu(df):
  # 0階・外れ値を NaN
  df.loc[df["room_kaisuu"] == 0, "room_kaisuu"] = np.nan
  df.loc[df["room_kaisuu"] > 100, "room_kaisuu"] = np.nan
  df.loc[df["room_kaisuu"] < -10, "room_kaisuu"] = np.nan
  # floor_count と同一の bin 設計
  bins = [-np.inf, 0, 1, 2, 3, 6, 10, 15, 20, 30, 40, 50, 60, np.inf]
  labels = [
    "B",
    "1", "2", "3", "4-6", "7-10",
    "11-15", "16-20", "21-30", "31-40",
    "41-50", "51-60", "60+"
  ]
  df["room_kaisuu_bin"] = pd.cut(
    df["room_kaisuu"],
    bins=bins,
    labels=labels
  )
  return df


#107 preprocess_snapshot_window_angleを東西南北でonehot
def preprocess_snapshot_window_angle(df):
  angle = df["snapshot_window_angle"]
  df["angle_north"] = angle.isin([1, 2, 8]).astype(int)
  df["angle_east"]  = angle.isin([3]).astype(int)
  df["angle_south"] = angle.isin([4, 5, 6]).astype(int)
  df["angle_west"]  = angle.isin([7]).astype(int)
  return df


#108 madori_number_allをbin化
def preprocess_madori_number_all(df):
  df.loc[df["madori_number_all"] < 1, "madori_number_all"] = np.nan
  df.loc[df["madori_number_all"] > 10, "madori_number_all"] = np.nan
  bins = [0, 1, 2, 3, 4, 5, 6, np.inf]
  labels = ["1", "2", "3", "4", "5", "6", "7+"]
  df["madori_number_bin"] = pd.cut(
    df["madori_number_all"],
    bins=bins,
    labels=labels
  )
  return df


#109 madori_kind_allをbin化
def preprocess_madori_kind_all(df):
  valid = [10, 20, 25, 30, 35, 40, 45, 50, 55]
  df.loc[~df["madori_kind_all"].isin(valid), "madori_kind_all"] = np.nan
  mapping = {
    "R":   [10],
    "K":   [20, 25],
    "DK":  [30, 35],
    "LDK": [40, 45, 50, 55],
  }
  df["madori_kind_bin"] = np.nan
  df["madori_kind_bin"] = pd.Series(index=df.index, dtype="object")
  for label, values in mapping.items():
    df.loc[df["madori_kind_all"].isin(values), "madori_kind_bin"] = label
  df["madori_kind_bin"] = df["madori_kind_bin"].astype(object)
  return df


#110 money_kyouekiの外れ値を処理。0円フラグの別カラムを作成。0円 or NaNフラグの別カラムを作成
def preprocess_money_kyoueki(df):
  df.loc[df["money_kyoueki"] > 200000, "money_kyoueki"] = np.nan
  df.loc[df["money_kyoueki"] < 0, "money_kyoueki"] = np.nan
  # 0フラグ
  df["flg_kyouekihi_0"] = (df["money_kyoueki"] == 0).astype(int)
  # 0 or NaN フラグ
  df["flg_kyouekihi_0_or_nan"] = (
    (df["money_kyoueki"].isna()) | (df["money_kyoueki"] == 0)
  ).astype(int)
  return df


#123 flg_no_parkingの4(駐車場無し)をフラグにした別カラムを作成。
def preprocess_parking_kubun(df):
  df["flg_no_parking"] = (df["parking_kubun"] == 4).astype(int)
  return df


#131 school_ele_distanceのNaNを中央値で埋め、対数変換した別カラムを作成
def preprocess_school_ele_distance(df):
  # 0以下は NaN, 9999以上は9999（念のため）
  df.loc[df["school_ele_distance"] <= 0, "school_ele_distance"] = np.nan
  df.loc[df["school_ele_distance"] >= 9999, "school_ele_distance"] = 9999
  # 中央値補完
  median = df["school_ele_distance"].median()
  df["school_ele_distance"] = df["school_ele_distance"].fillna(median)
  # log 特徴量追加
  df["school_ele_distance_log"] = np.log1p(df["school_ele_distance"])
  return df


#134 school_jun_distanceのNaNを中央値で埋め、対数変換した別カラムを作成
def preprocess_school_jun_distance(df):
  # 0以下は NaN, 9999以上は9999（念のため）
  df.loc[df["school_jun_distance"] <= 0, "school_jun_distance"] = np.nan
  df.loc[df["school_jun_distance"] >= 9999, "school_jun_distance"] = 9999
  # 中央値補完
  median = df["school_jun_distance"].median()
  df["school_jun_distance"] = df["school_jun_distance"].fillna(median)
  # log 特徴量追加
  df["school_jun_distance_log"] = np.log1p(df["school_jun_distance"])
  return df



#136 convenience_distanceのNaNを中央値で埋め、対数変換した別カラムを作成
def preprocess_convenience_distance(df):
  # 0以下は NaN, 9999以上は9999（念のため）
  df.loc[df["convenience_distance"] <= 0, "convenience_distance"] = np.nan
  df.loc[df["convenience_distance"] >= 9999, "convenience_distance"] = 9999
  # 中央値補完
  median = df["convenience_distance"].median()
  df["convenience_distance"] = df["convenience_distance"].fillna(median)
  # log 特徴量追加
  df["convenience_distance_log"] = np.log1p(df["convenience_distance"])
  return df


#137 super_distanceのNaNを中央値で埋め、対数変換した別カラムを作成
def preprocess_super_distance(df):
  # 0以下は NaN, 9999以上は9999（念のため）
  df.loc[df["super_distance"] <= 0, "super_distance"] = np.nan
  df.loc[df["super_distance"] >= 9999, "super_distance"] = 9999
  # 中央値補完
  median = df["super_distance"].median()
  df["super_distance"] = df["super_distance"].fillna(median)
  # log 特徴量追加
  df["super_distance_log"] = np.log1p(df["super_distance"])
  return df



# floor_count == room_kaisuu かつ 20 以上の最上階フラグを作成
def create_highrise_top_floor_flag(df):
  df["is_highrise_top_floor"] = 0
  cond = (
    df["floor_count"].notna() &
    df["room_kaisuu"].notna() &
    (df["floor_count"] >= 20) &
    (df["room_kaisuu"] == df["floor_count"])
  )
  df.loc[cond, "is_highrise_top_floor"] = 1
  return df


# 駅までの距離（log）と距離binを作成する
def create_station_distance(df):
  # GeoDataFrame 化
  gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["lon"], df["lat"]),
    crs="EPSG:4326"
  )
  # メートル座標系に変換
  gdf_m = gdf.to_crs(epsg=3857)
  stations_m = stations_data.to_crs(epsg=3857)
  # 最寄駅までの距離を計算
  nearest = gpd.sjoin_nearest(
    gdf_m,
    stations_m[["geometry"]],
    how="left",
    distance_col="station_distance"
  )
  # 異常値処理
  nearest.loc[nearest["station_distance"] <= 0, "station_distance"] = np.nan
  nearest.loc[nearest["station_distance"] >= 20000, "station_distance"] = 20000
  median = nearest["station_distance"].median()
  nearest["station_distance"] = nearest["station_distance"].fillna(median)
  # log 距離
  nearest["station_distance_log"] = np.log1p(nearest["station_distance"])
  # 距離 bin（人間の意味ベース）
  bins = [0, 100, 300, 500, 1000, 1500, 2000, 5000, 20000]
  nearest["station_distance_bin"] = pd.cut(
    nearest["station_distance"],
    bins=bins,
    labels=False,
    include_lowest=True
  )
  # df に戻す（生距離は残さない）
  df["station_distance_log"] = nearest["station_distance_log"].values
  df["station_distance_bin"] = nearest["station_distance_bin"].values
  return df


def create_landprice_knn(df, landprice_data, k=3):
  # ===== GeoDataFrame 化 =====
  gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["lon"], df["lat"]),
    crs="EPSG:4326"
  )
  # メートル系CRSへ
  gdf_m = gdf.to_crs(epsg=3857)
  landprice_m = landprice_data.to_crs(epsg=3857)
  # ===== 最近傍 k 点を取得 =====
  joined = gpd.sjoin_nearest(
    gdf_m,
    landprice_m[["L01_008", "geometry"]],
    how="left",
    distance_col="dist",
    max_distance=None
  )
  # ===== 距離0対策 =====
  joined["dist"] = joined["dist"].clip(lower=1.0)
  # ===== 距離加重平均（k点）=====
  # 同一物件（index）ごとに k 個使う
  joined["_w"] = 1.0 / joined["dist"]
  landprice_knn = (
    joined
    .groupby(joined.index)
    .apply(
      lambda x: np.average(
        x["L01_008"].iloc[:k],
        weights=x["_w"].iloc[:k]
      )
    )
  )
  # ===== df に戻す =====
  df["landprice_knn3"] = landprice_knn.values
  # 念のため異常値処理
  df["landprice_knn3"] = df["landprice_knn3"].clip(lower=1)
  # log 特徴量
  df["landprice_knn3_log"] = np.log1p(df["landprice_knn3"])
  return df


In [ ]:
def make_df(df):
  df = df.copy(deep = True)

  df = preprocess_building_status(df)       #4
  df = preprocess_building_type(df)         #7
  df = preprocess_unit_count(df)            #12
  df = preprocess_lon(df)                   #14
  df = preprocess_lat(df)                   #15
  df = preprocess_building_structure(df)    #16
  df = preprocess_total_floor_area(df)      #17
  df = preprocess_building_area(df)         #18
  df = preprocess_floor_count(df)           #19
  df = preprocess_basement_floor_count(df)  #20
  df = preprocess_year_built(df)            #21
  df = preprocess_building_land_area(df)    #22
  df = preprocess_building_land_chimoku(df) #26
  df = preprocess_land_youto(df)            #27
  df = preprocess_land_toshi(df)            #28
  df = preprocess_building_tag_id(df)       #45
  # df = preprocess_room_floor(df)            #49 → 106room_kaisuuと内容がかぶるため、コメントアウト
  df = preprocess_room_count(df)            #52
  df = preprocess_unit_area(df)             #53
  # df = preprocess_floor_plan_code(df)       #54 → 109madori_kind_allと内容がかぶるため、一旦コメントアウト
  df = preprocess_bukken_type(df)           #73
  df = preprocess_addr1(df)                 #79
  df = preprocess_flg_new(df)               #104
  df = preprocess_room_kaisuu(df)           #106
  df = preprocess_snapshot_window_angle(df) #107
  df = preprocess_madori_number_all(df)     #108
  df = preprocess_madori_kind_all(df)       #109
  df = preprocess_money_kyoueki(df)         #110
  df = preprocess_parking_kubun(df)         #123
  df = preprocess_school_ele_distance(df)   #131
  df = preprocess_school_jun_distance(df)   #134
  df = preprocess_convenience_distance(df)  #136
  df = preprocess_super_distance(df)        #137

  df = create_highrise_top_floor_flag(df)
  df = create_station_distance(df)
  df = create_landprice_knn(df, landprice_data)

  df = drop_after_feature_engineering(df)

  # objectになりうるカラムをすべてcategory化
  CAT_COLS = [
    "lon_mesh",
    "lat_mesh",
    "land_youto",
    "land_toshi",
    "addr1_1",
    "pref_city",
    "madori_kind_bin",
    "floor_count_bin",
    "room_kaisuu_bin",
    "madori_number_bin",
    "year_built_bin",
    "building_land_area_bin",
  ]
  df = df.copy()
  df[CAT_COLS] = df[CAT_COLS].astype("category")

  return df

# 前処理② テストデータ作成

In [ ]:
df_t = make_df(df)

# 特徴量エンジニアリングの結果確認用

In [ ]:
df_kakunin = make_df(df)

In [ ]:
df_kakunin.columns.tolist()

In [ ]:
df_kakunin.dtypes.value_counts()

In [ ]:
df_kakunin.select_dtypes(include="object").columns

In [ ]:
df_kakunin.isna().mean().sort_values(ascending=False).head(50)

In [ ]:
tag_cols = [c for c in df_kakunin.columns if c.startswith("tag_")]
df_kakunin[tag_cols].sum().sort_values(ascending=False).head(20)

In [ ]:
bin_cols = [c for c in df_kakunin.columns if "bin" in c]
df_kakunin[bin_cols].describe()

In [ ]:
df_kakunin.select_dtypes(include="number").corr()["money_room"] \
  .sort_values(ascending=False).head(20)

# モデル作成

In [ ]:
# ===== データ準備 =====

df_train = df_t.copy(deep = True)

print(df_train.shape)
df_train.head()


In [ ]:
# 目的変数があるか
assert "money_room" in df_train.columns

# object 型が残ってないか（category はOK）
df_train.dtypes.value_counts()


In [ ]:
# ===== X, y 分割 =====
X = df_train.drop(columns=["money_room"])
y = df_train["money_room"]

# ホールドアウト（まずはシンプルに）
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


In [ ]:
# ===== モデル定義 =====
model = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

In [ ]:
model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric="rmse",
    callbacks=[lgb.log_evaluation(50)]
)

In [ ]:
# ===== 簡易評価 =====
y_pred = model.predict(X_valid)
rmse = np.sqrt(mean_squared_error(y_valid, y_pred))

print(f"Validation RMSE: {rmse:.4f}")

In [ ]:
def mape(y_true, y_pred):
    return 100 * np.mean(np.abs((y_true - y_pred) / y_true))

# 予測
y_valid_pred = model.predict(X_valid)

# スコア計算
score = mape(y_valid, y_valid_pred)
print(f"Validation MAPE: {score:.4f}")


#目的変数の対数変換込みでの学習

In [ ]:
# ===== データ準備 =====

df_train = df_t.copy(deep = True)

print(df_train.shape)
df_train.head()

In [ ]:
X = df_train.drop(columns=[TARGET])
y = df_train[TARGET]

# log変換（0対策で log1p）
y_log = np.log1p(y)

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)


In [ ]:
model = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric="rmse",
    callbacks=[lgb.log_evaluation(50)]
)


In [ ]:
# log → 元スケールに戻す
y_valid_pred_log = model.predict(X_valid)
y_valid_pred = np.expm1(y_valid_pred_log)
y_valid_true = np.expm1(y_valid)

# 0割・異常値対策
y_valid_pred = np.clip(y_valid_pred, 1, None)

# MAPE
mape = np.mean(np.abs((y_valid_true - y_valid_pred) / y_valid_true)) * 100
print(f"Validation MAPE: {mape:.4f}")


In [ ]:
mask = y_valid_true < 10000000  # 1000万円未満
mape_low = np.mean(
    np.abs((y_valid_true[mask] - y_valid_pred[mask]) / y_valid_true[mask])
) * 100

print(f"Low price MAPE: {mape_low:.2f}")


In [ ]:
# mask作成
mask1 = y_valid_true < 15_000_000
mask2 = (y_valid_true >= 15_000_000) & (y_valid_true < 30_000_000)
mask3 = y_valid_true >= 30_000_000

# MAPE計算
mape1 = np.mean(np.abs((y_valid_true[mask1] - y_valid_pred[mask1]) / y_valid_true[mask1])) * 100
mape2 = np.mean(np.abs((y_valid_true[mask2] - y_valid_pred[mask2]) / y_valid_true[mask2])) * 100
mape3 = np.mean(np.abs((y_valid_true[mask3] - y_valid_pred[mask3]) / y_valid_true[mask3])) * 100

print(f"~1500万 MAPE: {mape1:.2f}")
print(f"1500万~3000万 MAPE: {mape2:.2f}")
print(f"3000万~ MAPE: {mape3:.2f}")

In [ ]:
# feature importance

def get_feature_importance(model, X):
  fi = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
  }).sort_values("importance", ascending=False)
  return fi

def get_feature_importance_gain(model, X):
  fi = pd.DataFrame({
    "feature": X.columns,
    "importance": model.booster_.feature_importance(importance_type="gain")
  }).sort_values("importance", ascending=False)
  return fi

In [ ]:
#split回数(使われた回数)
fi = get_feature_importance(model, X_train)
print(fi.head(20))

In [ ]:
#gain(どれだけ損失を減らしたか)
fi = get_feature_importance_gain(model, X_train)
print(fi.head(20))

In [ ]:
# =========================
# log → 元スケール
# =========================
y_valid_pred_log = model.predict(X_valid)

y_valid_pred = np.expm1(y_valid_pred_log)
y_valid_true = np.expm1(y_valid)

# 0割・異常値対策
y_valid_pred = np.clip(y_valid_pred, 1, None)

# =========================
# DataFrame 作成
# =========================
df_valid = pd.DataFrame({
  "price": y_valid_true,
  "pred": y_valid_pred
})

df_valid["ape"] = np.abs(
  (df_valid["price"] - df_valid["pred"]) / df_valid["price"]
)

# =========================
# 価格順ソート
# =========================
df_valid = df_valid.sort_values("price").reset_index(drop=True)

# =========================
# 0.5% window MAPE
# =========================
N = len(df_valid)
window_size = max(int(N * 0.005), 30)

window_mape = []
window_center_price = []

for start in range(0, N - window_size + 1):
  end = start + window_size
  window = df_valid.iloc[start:end]

  mape = window["ape"].mean() * 100
  center_price = window["price"].median()

  window_mape.append(mape)
  window_center_price.append(center_price)

df_valid_mape_curve = pd.DataFrame({
  "price_center": window_center_price,
  "mape": window_mape
})

# =========================
# プロット
# =========================
plt.figure(figsize=(10, 5))
plt.plot(df_valid_mape_curve["price_center"],
         df_valid_mape_curve["mape"])
plt.xlabel("Price")
plt.ylabel("MAPE (%)")
plt.title("Validation MAPE curve (0.5% window)")
plt.grid(True)
plt.show()


# Two-stageで実装

In [ ]:
#OOF

# ===== データ準備 =====
df_train = df_t.copy(deep = True)

TARGET = "money_room"
X = df_train.drop(columns=[TARGET])
y = df_train[TARGET]
y_log = np.log1p(y)

# ===== OOF 用意 =====
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

oof_pred_log = np.zeros(len(X))

# ===== CV ループ =====
for fold, (train_idx, valid_idx) in enumerate(kf.split(X)):
  print(f"\n===== Fold {fold + 1} =====")

  X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
  y_train, y_valid = y_log.iloc[train_idx], y_log.iloc[valid_idx]

  model = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
  )

  model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric="rmse",
    callbacks=[lgb.log_evaluation(50)]
  )

  # valid 部分だけ予測して書き戻す
  oof_pred_log[valid_idx] = model.predict(X_valid)

# ===== OOF MAPE 計算 =====
y_true = y.values
y_pred = np.expm1(oof_pred_log)

# 0割・異常値対策
y_pred = np.clip(y_pred, 1, None)

mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
print(f"\nOOF MAPE: {mape:.4f}")


In [ ]:
#プロットで境界線を確認する(あとで自動化するけど)

# ===== 前提 =====
# y             : 元スケールの正解値（money_room）
# oof_pred_log  : log1p スケールの OOF 予測

y_true = y.values
y_pred = np.expm1(oof_pred_log)

# 安全対策
y_pred = np.clip(y_pred, 1, None)

# ===== 誤差計算 =====
log_price = np.log1p(y_true)
ape = np.abs(y_pred - y_true) / y_true

# ===== log価格でソート =====
order = np.argsort(log_price)
log_price_sorted = log_price[order]
ape_sorted = ape[order]

# ===== 移動平均 =====
window = int(len(ape_sorted) * 0.07)  # 全体の7%
window = max(window, 10)

ape_ma = np.convolve(
  ape_sorted,
  np.ones(window) / window,
  mode="valid"
)

log_price_ma = log_price_sorted[:len(ape_ma)]

# ===== プロット =====
plt.figure()
plt.scatter(log_price_sorted, ape_sorted, alpha=0.3)
plt.plot(log_price_ma, ape_ma)
plt.xlabel("log1p(money_room)")
plt.ylabel("Absolute Percentage Error (APE)")
plt.title("OOF Error Distribution")
plt.show()


In [ ]:
# ===== 境界抽出 =====

y_true = y.values
y_pred = np.expm1(oof_pred_log)
y_pred = np.clip(y_pred, 1, None)

log_price = np.log1p(y_true)
ape = np.abs(y_true - y_pred) / y_true

# ===== 誤差DF作成 =====
df_err = pd.DataFrame({
  "log_price": log_price,
  "ape": ape
}).sort_values("log_price").reset_index(drop=True)

# ===== 移動平均 =====
window_ratio = 0.02
window = max(10, int(len(df_err) * window_ratio))

df_err["ape_ma"] = (
  df_err["ape"]
  .rolling(window=window, center=True)
  .mean()
)

# ===== 勾配 =====
df_err["slope"] = np.gradient(df_err["ape_ma"])

# ===== 境界決定 =====
def extract_boundaries(df_err, K):
  """
  df_err : log_price, ape_ma, slope を含む DataFrame
  K      : 分割数
  """
  df_tmp = df_err.dropna(subset=["slope"]).copy()
  df_tmp["abs_slope"] = np.abs(df_tmp["slope"])

  # 勾配が大きい順
  df_tmp = df_tmp.sort_values("abs_slope", ascending=False)

  # 上位 K-1 個を境界に
  boundaries = np.sort(
    df_tmp["log_price"].values[:K-1]
  )
  return boundaries

K = 2 # ラベル数。2以上。
boundaries = extract_boundaries(df_err, K)

y_bin = np.digitize(df_err["log_price"].values, boundaries)


In [ ]:
# クラス分類(p1 + p2 + p3 + …… pn = 1)
K = len(np.unique(y_bin))

X_train, X_valid, y_train, y_valid = train_test_split(
  X,
  y_bin,
  test_size=0.2,
  random_state=42,
  stratify=y_bin
)

clf = lgb.LGBMClassifier(
  objective="multiclass",
  num_class=K,
  n_estimators=300,
  learning_rate=0.05,
  subsample=0.8,
  colsample_bytree=0.8,
  random_state=42,
  n_jobs=-1
)

clf.fit(
  X_train,
  y_train,
  eval_set=[(X_valid, y_valid)],
  eval_metric="multi_logloss",
  callbacks=[lgb.log_evaluation(50)]
)

proba_valid = clf.predict_proba(X_valid)


In [ ]:
# 回帰モデル構築

reg_models = {}

for k in range(K):
  idx = (y_bin == k)

  X_k = X[idx]
  y_k = y_log[idx]

  print(f"bin {k}: samples = {len(X_k)}")

  reg = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
  )

  reg.fit(X_k, y_k)

  reg_models[k] = reg


In [ ]:
# Stage1 の確率（全データ）
proba = clf.predict_proba(X)  # (N, K)

# 最終予測
y_pred_final = np.zeros(len(X))

for k in range(K):
  y_pred_k_log = reg_models[k].predict(X)
  y_pred_k = np.expm1(y_pred_k_log)

  y_pred_final += proba[:, k] * y_pred_k

y_pred_final = np.clip(y_pred_final, 1, None)


In [ ]:
mape = np.mean(np.abs((y.values - y_pred_final) / y.values)) * 100
print(f"Two-stage MAPE: {mape:.4f}")

In [ ]:
models = []

for k in range(K):
  idx = (y_bin == k)

  model = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
  )

  model.fit(
    X.loc[idx],
    np.log1p(y_true[idx])
  )

  models.append(model)

y_bin_pred = clf.predict(X)

y_pred_two_stage = np.zeros(len(X))

for k in range(K):
  idx = (y_bin_pred == k)
  if idx.sum() == 0:
    continue

  y_pred_two_stage[idx] = np.expm1(
    models[k].predict(X.loc[idx])
  )

y_pred_two_stage = np.clip(y_pred_two_stage, 1, None)

LOW_BIN = 0
mask_low = (y_bin == LOW_BIN)

low_mape = (
  np.mean(
    np.abs(
      (y_true[mask_low] - y_pred_two_stage[mask_low])
      / y_true[mask_low]
    )
  ) * 100
)

print(f"LowPrice MAPE: {low_mape:.4f}")
print("True bin counts :", np.bincount(y_bin))
print("Pred bin counts :", np.bincount(y_bin_pred))

In [ ]:
# 確率重み付け

# shape: (n_samples, K)
preds = np.zeros((len(X), K))

for k in range(K):
  preds[:, k] = np.expm1(
    models[k].predict(X)
  )

y_pred_soft = np.sum(proba * preds, axis=1)
y_pred_soft = np.clip(y_pred_soft, 1, None)

mape_all = (
  np.mean(
    np.abs((y_true - y_pred_soft) / y_true)
  ) * 100
)

print(f"Soft Two-stage MAPE: {mape_all:.4f}")

LOW_BIN = 0
mask_low = (y_bin == LOW_BIN)

low_mape = (
  np.mean(
    np.abs(
      (y_true[mask_low] - y_pred_soft[mask_low])
      / y_true[mask_low]
    )
  ) * 100
)

print(f"LowPrice MAPE: {low_mape:.4f}")



In [ ]:
print("Mean p_low:", proba[:, 0].mean())
print("LowPrice mean p_low:", proba[mask_low, 0].mean())


In [ ]:
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

oof_pred_soft = np.zeros(len(X))

for fold, (train_idx, valid_idx) in enumerate(kf.split(X)):
  print(f"\n===== Fold {fold + 1} =====")

  X_tr, X_va = X.iloc[train_idx], X.iloc[valid_idx]
  y_tr, y_va = y_true[train_idx], y_true[valid_idx]

  y_bin_tr = y_bin[train_idx]

  # ===== 1. 分類器 =====
  clf = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=K,
    n_estimators=200,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
  )

  clf.fit(X_tr, y_bin_tr)

  proba_va = clf.predict_proba(X_va)  # (n_valid, K)

  # ===== 2. bin別回帰 =====
  models = []

  for k in range(K):
    idx_k = (y_bin_tr == k)

    model = lgb.LGBMRegressor(
      n_estimators=200,
      learning_rate=0.05,
      subsample=0.8,
      colsample_bytree=0.8,
      random_state=42,
      n_jobs=-1
    )

    model.fit(
      X_tr.loc[idx_k],
      np.log1p(y_tr[idx_k])
    )

    models.append(model)

  # ===== 3. soft 合成 =====
  preds = np.zeros((len(valid_idx), K))

  for k in range(K):
    preds[:, k] = np.expm1(
      models[k].predict(X_va)
    )

  y_pred_va = np.sum(proba_va * preds, axis=1)
  y_pred_va = np.clip(y_pred_va, 1, None)

  oof_pred_soft[valid_idx] = y_pred_va

# ===== OOF MAPE =====
mape = (
  np.mean(
    np.abs((y_true - oof_pred_soft) / y_true)
  ) * 100
)

print(f"\nOOF Soft Two-stage MAPE: {mape:.4f}")

# ===== LowPrice MAPE =====
LOW_BIN = 0
mask_low = (y_bin == LOW_BIN)

low_mape = (
  np.mean(
    np.abs(
      (y_true[mask_low] - oof_pred_soft[mask_low])
      / y_true[mask_low]
    )
  ) * 100
)

print(f"LowPrice OOF MAPE: {low_mape:.4f}")


# Two-Stageで実装②

In [ ]:
#OOF onw-stageのMAPEをプロット

# =========================
# 前処理
# =========================
df_train = df_t.copy(deep = True)

X = df_train.drop(columns=[TARGET])
y = df_train[TARGET].values

# log変換
y_log = np.log1p(y)

# =========================
# OOF 用意
# =========================
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

oof_pred_log = np.zeros(len(X))

# =========================
# モデル定義
# =========================
model_params = dict(
  n_estimators=300,
  learning_rate=0.05,
  subsample=0.8,
  colsample_bytree=0.8,
  random_state=42,
  n_jobs=-1
)

# =========================
# OOF 学習
# =========================
for fold, (tr_idx, va_idx) in enumerate(kf.split(X)):
  print(f"Fold {fold + 1}")

  X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
  y_tr, y_va = y_log[tr_idx], y_log[va_idx]

  model = lgb.LGBMRegressor(**model_params)

  model.fit(
    X_tr, y_tr,
    eval_set=[(X_va, y_va)],
    eval_metric="rmse",
    callbacks=[lgb.log_evaluation(100)]
  )

  oof_pred_log[va_idx] = model.predict(X_va)

# =========================
# log → 元スケール
# =========================
y_oof_pred = np.expm1(oof_pred_log)
y_true = y.copy()

# 0割・異常値対策
y_oof_pred = np.clip(y_oof_pred, 1, None)

# =========================
# OOF結果 DataFrame
# =========================
df_oof = pd.DataFrame({
  "price": y_true,
  "pred": y_oof_pred
})

df_oof["ape"] = np.abs((df_oof["price"] - df_oof["pred"]) / df_oof["price"])

# =========================
# 価格順にソート
# =========================
df_oof = df_oof.sort_values("price").reset_index(drop=True)

# =========================
# 0.5% window MAPE
# =========================
N = len(df_oof)
window_size = max(int(N * 0.005), 30)  # 最低30件

window_mape = []
window_center_price = []

for start in range(0, N - window_size + 1):
  end = start + window_size
  window = df_oof.iloc[start:end]

  mape = window["ape"].mean() * 100
  center_price = window["price"].median()

  window_mape.append(mape)
  window_center_price.append(center_price)

df_mape_curve = pd.DataFrame({
  "price_center": window_center_price,
  "mape": window_mape
})

# =========================
# プロット
# =========================
plt.figure(figsize=(10, 5))
plt.plot(df_mape_curve["price_center"], df_mape_curve["mape"])
plt.xlabel("Price")
plt.ylabel("MAPE (%)")
plt.title("OOF MAPE curve (0.5% window)")
plt.grid(True)
plt.show()


In [ ]:
# =========================
# 前処理
# =========================
df_train = df_t.copy(deep = True)

X = df_train.drop(columns=[TARGET])
y = df_train[TARGET].values

In [ ]:
# two-stage sigmoid OOF

# ======================
# 設定
# ======================

n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

params = {
  "n_estimators": 1000,
  "learning_rate": 0.05,
  "num_leaves": 31,
  "random_state": 42,
  "n_jobs": -1
}

# μ と s は適宜調整
mu1 = 20000000
mu2 = 60000000
s = 10000000

def sigmoid(x):
  return 1 / (1 + np.exp(-x))


# ======================
# Step1: one-stage OOF
# ======================

oof_base = np.zeros(len(X))

for tr_idx, va_idx in kf.split(X):

  X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
  y_tr, y_va = y[tr_idx], y[va_idx]

  model = LGBMRegressor(**params)
  model.fit(X_tr, y_tr)

  oof_base[va_idx] = model.predict(X_va)


# ======================
# Step2: p 作成
# ======================

p_low  = sigmoid((mu1 - oof_base) / s)
p_high = sigmoid((oof_base - mu2) / s)
p_mid  = 1 - p_low - p_high

# 安全のためclip
p_low  = np.clip(p_low,  0, 1)
p_mid  = np.clip(p_mid,  0, 1)
p_high = np.clip(p_high, 0, 1)


# ======================
# Step3: 3モデル OOF
# ======================

final_oof = np.zeros(len(X))

for tr_idx, va_idx in kf.split(X):

  X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
  y_tr, y_va = y[tr_idx], y[va_idx]

  # train側の重み
  w_low_tr  = p_low[tr_idx]
  w_mid_tr  = p_mid[tr_idx]
  w_high_tr = p_high[tr_idx]

  # モデル
  model_low  = LGBMRegressor(**params)
  model_mid  = LGBMRegressor(**params)
  model_high = LGBMRegressor(**params)

  # 学習
  model_low.fit(X_tr, y_tr, sample_weight=w_low_tr)
  model_mid.fit(X_tr, y_tr, sample_weight=w_mid_tr)
  model_high.fit(X_tr, y_tr, sample_weight=w_high_tr)

  # 予測
  pred_low  = model_low.predict(X_va)
  pred_mid  = model_mid.predict(X_va)
  pred_high = model_high.predict(X_va)

  # val側の重みで合成
  final_oof[va_idx] = (
    p_low[va_idx]  * pred_low +
    p_mid[va_idx]  * pred_mid +
    p_high[va_idx] * pred_high
  )


# ======================
# Step4: MAPE
# ======================

mape = mean_absolute_percentage_error(y, final_oof) * 100
print(f"Final OOF MAPE: {mape:.4f}%")


In [ ]:
import numpy as np

def mape(y_true, y_pred):
  return np.mean(np.abs((y_true - y_pred) / y_true)) * 100


# 真の価格でマスク作成
mask_low  = y < mu1
mask_mid  = (y >= mu1) & (y < mu2)
mask_high = y >= mu2

# one-stageのOOFもあるなら比較できる
print("=== One-stage ===")
print(f"Low  MAPE: {mape(y[mask_low],  oof_base[mask_low]):.4f}%")
print(f"Mid  MAPE: {mape(y[mask_mid],  oof_base[mask_mid]):.4f}%")
print(f"High MAPE: {mape(y[mask_high], oof_base[mask_high]):.4f}%")

print("\n=== MoE ===")
print(f"Low  MAPE: {mape(y[mask_low],  final_oof[mask_low]):.4f}%")
print(f"Mid  MAPE: {mape(y[mask_mid],  final_oof[mask_mid]):.4f}%")
print(f"High MAPE: {mape(y[mask_high], final_oof[mask_high]):.4f}%")


In [ ]:
# =========================
# two-stage OOF結果 DataFrame
# =========================

y_true = y.copy()
y_oof_pred = final_oof.copy()

# 0割・異常値対策
y_oof_pred = np.clip(y_oof_pred, 1, None)

df_oof = pd.DataFrame({
  "price": y_true,
  "pred": y_oof_pred
})

df_oof["ape"] = np.abs((df_oof["price"] - df_oof["pred"]) / df_oof["price"])

# =========================
# 価格順にソート
# =========================
df_oof = df_oof.sort_values("price").reset_index(drop=True)

# =========================
# 0.5% window MAPE
# =========================
N = len(df_oof)
window_size = max(int(N * 0.005), 30)

window_mape = []
window_center_price = []

for start in range(0, N - window_size + 1):
  end = start + window_size
  window = df_oof.iloc[start:end]

  mape = window["ape"].mean() * 100
  center_price = window["price"].median()

  window_mape.append(mape)
  window_center_price.append(center_price)

df_mape_curve_moe = pd.DataFrame({
  "price_center": window_center_price,
  "mape": window_mape
})

# =========================
# プロット
# =========================
plt.figure(figsize=(10, 5))
plt.plot(df_mape_curve_moe["price_center"], df_mape_curve_moe["mape"])
plt.xlabel("Price")
plt.ylabel("MAPE (%)")
plt.title("Two-stage OOF MAPE curve (0.5% window)")
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(df_mape_curve["price_center"],
         df_mape_curve["mape"],
         label="One-stage")

plt.plot(df_mape_curve_moe["price_center"],
         df_mape_curve_moe["mape"],
         label="Two-stage (MoE)")

plt.xlabel("Price")
plt.ylabel("MAPE (%)")
plt.title("MAPE curve comparison (0.5% window)")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
df_oof[df_oof["price"] < 20000000].shape


# softmax, sigmoidの比較

In [ ]:
# 関数部分

def sigmoid(x):
  return 1 / (1 + np.exp(-x))


def run_moe(X, y, kf, params, mu1, mu2, s=None, gate_type="sigmoid"):

  n = len(X)

  # =========================
  # Step1: gating確率作成
  # =========================

  if gate_type == "sigmoid":

    # one-stage OOF
    oof_base = np.zeros(n)

    for tr_idx, va_idx in kf.split(X):
      model = LGBMRegressor(**params)
      model.fit(X.iloc[tr_idx], y[tr_idx])
      oof_base[va_idx] = model.predict(X.iloc[va_idx])

    # 累積型sigmoid
    p_low_cum  = sigmoid((mu1 - oof_base) / s)
    p_low_mid_cum = sigmoid((mu2 - oof_base) / s)

    p_low  = p_low_cum
    p_mid  = p_low_mid_cum - p_low_cum
    p_high = 1 - p_low_mid_cum

  elif gate_type == "softmax":

    # クラス作成
    y_class = np.zeros(n)
    y_class[y < mu1] = 0
    y_class[(y >= mu1) & (y < mu2)] = 1
    y_class[y >= mu2] = 2

    oof_proba = np.zeros((n, 3))

    for tr_idx, va_idx in kf.split(X):

      clf = LGBMClassifier(
        objective="multiclass",
        num_class=3,
        n_estimators=params["n_estimators"],
        learning_rate=params["learning_rate"],
        random_state=42,
        n_jobs=-1
      )

      clf.fit(X.iloc[tr_idx], y_class[tr_idx])
      oof_proba[va_idx] = clf.predict_proba(X.iloc[va_idx])

    p_low  = oof_proba[:, 0]
    p_mid  = oof_proba[:, 1]
    p_high = oof_proba[:, 2]

  else:
    raise ValueError("gate_type must be 'sigmoid' or 'softmax'")

  # 安全クリップ
  p_low  = np.clip(p_low,  0, 1)
  p_mid  = np.clip(p_mid,  0, 1)
  p_high = np.clip(p_high, 0, 1)

  # =========================
  # Step2: MoE回帰
  # =========================

  final_oof = np.zeros(n)

  for tr_idx, va_idx in kf.split(X):

    model_low  = LGBMRegressor(**params)
    model_mid  = LGBMRegressor(**params)
    model_high = LGBMRegressor(**params)

    model_low.fit(X.iloc[tr_idx], y[tr_idx], sample_weight=p_low[tr_idx])
    model_mid.fit(X.iloc[tr_idx], y[tr_idx], sample_weight=p_mid[tr_idx])
    model_high.fit(X.iloc[tr_idx], y[tr_idx], sample_weight=p_high[tr_idx])

    pred_low  = model_low.predict(X.iloc[va_idx])
    pred_mid  = model_mid.predict(X.iloc[va_idx])
    pred_high = model_high.predict(X.iloc[va_idx])

    final_oof[va_idx] = (
      p_low[va_idx]  * pred_low +
      p_mid[va_idx]  * pred_mid +
      p_high[va_idx] * pred_high
    )

  mape = mean_absolute_percentage_error(y, final_oof) * 100

  return mape, final_oof


In [ ]:
# 前処理、パラメータ設定

df_train = df_t.copy(deep = True)
X = df_train.drop(columns=[TARGET])
y = df_train[TARGET].values

n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

params = {
  "n_estimators": 1000,
  "learning_rate": 0.05,
  "num_leaves": 31,
  "random_state": 42,
  "n_jobs": -1
}

mu1 = 25000000,
mu2 = 60000000,
s = 10000000,

In [ ]:
# sigmoid関数
mape_sig, pred_sig = run_moe(
  X, y, kf, params,
  mu1 = mu1,
  mu2 = mu2,
  s = s,
  gate_type = "sigmoid"
)

print("Sigmoid MAPE:", mape_sig)


In [ ]:
# softmax関数
mape_soft, pred_soft = run_moe(
  X, y, kf, params,
  mu1 = mu1,
  mu2 = mu2,
  gate_type = "softmax"
)

print("Softmax MAPE:", mape_soft)


In [ ]:
def make_mape_curve(y_true, y_pred, window_ratio=0.005, min_size=30):

  df_tmp = pd.DataFrame({
    "price": y_true,
    "pred": y_pred
  })

  df_tmp["ape"] = np.abs((df_tmp["price"] - df_tmp["pred"]) / df_tmp["price"])

  df_tmp = df_tmp.sort_values("price").reset_index(drop=True)

  N = len(df_tmp)
  window_size = max(int(N * window_ratio), min_size)

  window_mape = []
  window_center_price = []

  for start in range(0, N - window_size + 1):
    end = start + window_size
    window = df_tmp.iloc[start:end]

    mape = window["ape"].mean() * 100
    center_price = window["price"].median()

    window_mape.append(mape)
    window_center_price.append(center_price)

  return pd.DataFrame({
    "price_center": window_center_price,
    "mape": window_mape
  })

# df_curve_one  = make_mape_curve(y, pred_one)
df_curve_sig  = make_mape_curve(y, pred_sig)
df_curve_soft = make_mape_curve(y, pred_soft)

plt.figure(figsize=(10, 5))

# plt.plot(df_curve_one["price_center"],
#          df_curve_one["mape"],
#          label="One-stage")

plt.plot(df_curve_sig["price_center"],
         df_curve_sig["mape"],
         label="Two-stage (Sigmoid)")

plt.plot(df_curve_soft["price_center"],
         df_curve_soft["mape"],
         label="Two-stage (Softmax)")

plt.xlabel("Price")
plt.ylabel("MAPE (%)")
plt.title("MAPE curve comparison (0.5% window)")
plt.legend()
plt.grid(True)
plt.show()


# 決定木によって分割点探索

In [ ]:
# 分割点を決定木によって確認する


# ======================
# 0. データ準備
# ======================

df_train = df_t.copy(deep=True)

TARGET = "money_room"

X = df_train.drop(columns=[TARGET])
y = df_train[TARGET].values


# ======================
# 1. one-stage OOF
# ======================

n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

params = {
  "n_estimators": 1000,
  "learning_rate": 0.05,
  "num_leaves": 31,
  "random_state": 42,
  "n_jobs": -1
}

oof_pred = np.zeros(len(X))

for tr_idx, va_idx in kf.split(X):

  X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
  y_tr = y[tr_idx]

  model = LGBMRegressor(**params)
  model.fit(X_tr, y_tr)

  oof_pred[va_idx] = model.predict(X_va)


# ======================
# 2. one-stage MAPE
# ======================

y_clip = np.clip(y, 1, None)
mape_one = mean_absolute_percentage_error(y_clip, oof_pred) * 100
print(f"One-stage OOF MAPE: {mape_one:.4f}%")


# ======================
# 3. 残差率作成
# ======================

residual_rate = np.abs(y_clip - oof_pred) / y_clip


# ======================
# 4. 価格のみで決定木（μ推定）
# ======================

X_price = y_clip.reshape(-1, 1)

tree = DecisionTreeRegressor(
  max_depth=2,           # 3領域作る
  min_samples_leaf=200,  # 過学習防止（調整可）
  random_state=42
)

tree.fit(X_price, residual_rate)

thresholds = tree.tree_.threshold
thresholds = thresholds[thresholds > 0]

mu_candidates = sorted(thresholds)

print("\n推定された価格分割点（μ候補）:")
for mu in mu_candidates:
  print(f"{mu:,.0f} 円")


# ======================
# 5. 各領域の誤差確認
# ======================

region_pred = tree.predict(X_price)

regions = []

for r in np.unique(region_pred):
  mask = region_pred == r
  regions.append((
    y_clip[mask].mean(),
    residual_rate[mask].mean(),
    mask.sum()
  ))

regions = sorted(regions, key=lambda x: x[0])

print("\n各価格帯の平均価格・平均誤差率・件数")
for avg_price, avg_err, count in regions:
  print(f"平均価格: {avg_price:,.0f}円 | 残差率: {avg_err:.4f} | 件数: {count}")


# ベイズ最適化、optunaによる分割点探索

In [ ]:
# ===============================
# 特徴量・ターゲット定義
# ===============================
df_train = df_t.copy(deep=True)
target = "money_room"
features = [c for c in df_train.columns if c != target]

X = df_train[features]
y = df_train[target].values

# ===============================
# パラメータ（軽量）
# ===============================
params = {
  "n_estimators": 300,
  "learning_rate": 0.05,
  "num_leaves": 31,
  "random_state": 42,
  "n_jobs": -1
}

# ===============================
# MoE関数（softmaxゲート）
# ===============================
def run_moe_softmax(X, y, mu1, mu2):

  n = len(X)
  kf = KFold(n_splits=3, shuffle=True, random_state=42)

  # --- クラス作成 ---
  y_class = np.zeros(n)
  y_class[y < mu1] = 0
  y_class[(y >= mu1) & (y < mu2)] = 1
  y_class[y >= mu2] = 2

  oof_proba = np.zeros((n, 3))
  final_oof = np.zeros(n)

  # ======================
  # Step1: Gating学習
  # ======================
  for tr_idx, va_idx in kf.split(X):

    clf = LGBMClassifier(
      objective="multiclass",
      num_class=3,
      n_estimators=200,
      learning_rate=0.05,
      random_state=42,
      n_jobs=-1
    )

    clf.fit(X.iloc[tr_idx], y_class[tr_idx])
    oof_proba[va_idx] = clf.predict_proba(X.iloc[va_idx])

  p_low  = oof_proba[:, 0]
  p_mid  = oof_proba[:, 1]
  p_high = oof_proba[:, 2]

  # ======================
  # Step2: 専門家回帰
  # ======================
  for tr_idx, va_idx in kf.split(X):

    model_low  = LGBMRegressor(**params)
    model_mid  = LGBMRegressor(**params)
    model_high = LGBMRegressor(**params)

    model_low.fit(X.iloc[tr_idx], y[tr_idx], sample_weight=p_low[tr_idx])
    model_mid.fit(X.iloc[tr_idx], y[tr_idx], sample_weight=p_mid[tr_idx])
    model_high.fit(X.iloc[tr_idx], y[tr_idx], sample_weight=p_high[tr_idx])

    pred_low  = model_low.predict(X.iloc[va_idx])
    pred_mid  = model_mid.predict(X.iloc[va_idx])
    pred_high = model_high.predict(X.iloc[va_idx])

    final_oof[va_idx] = (
      p_low[va_idx]  * pred_low +
      p_mid[va_idx]  * pred_mid +
      p_high[va_idx] * pred_high
    )

  mape = mean_absolute_percentage_error(y, final_oof) * 100

  return mape

In [ ]:
# ===============================
# 軽量サブサンプル（5万件）
# ===============================
df_sub = df_train.sample(n=50000, random_state=42)

X_sub = df_sub[features]
y_sub = df_sub[target].values

# ===============================
# objective
# ===============================
def objective(trial):

  mu1 = trial.suggest_int("mu1", 5_000_000, 25_000_000)
  mu2 = trial.suggest_int("mu2", 30_000_000, 80_000_000)

  if mu1 >= mu2:
    return 999

  mape = run_moe_softmax(X_sub, y_sub, mu1, mu2)
  return mape

# ===============================
# 実行
# ===============================
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=25)

print("Best params:", study.best_params)
print("Best MAPE:", study.best_value)

In [ ]:
df_train = df_t.copy(deep=True)

X = df_train.drop(columns=[TARGET])
y = df_train[TARGET].values

n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

params = {
  "n_estimators": 1000,
  "learning_rate": 0.05,
  "num_leaves": 31,
  "random_state": 42,
  "n_jobs": -1
}


# =========================
# 探索範囲
# =========================
# mu1_list = [8_000_000, 10_000_000, 12_000_000, 15_000_000]
# mu2_list = [15_000_000, 20_000_000, 25_000_000, 30_000_000]
mu1_list = [15_000_000]
mu2_list = [15_000_000, 20_000_000, 25_000_000, 30_000_000]

best_mape = np.inf
best_mu1 = None
best_mu2 = None

all_results = []

# =========================
# グリッド探索
# =========================
for mu1 in mu1_list:
  for mu2 in mu2_list:

    if mu1 >= mu2:
      continue

    print("=" * 60)
    print(f"Running mu1={mu1:,}, mu2={mu2:,}")

    try:
      mape, _ = run_moe(
        X,
        y,
        kf,
        params,
        mu1=mu1,
        mu2=mu2,
        s=None,                # softmaxなので不要
        gate_type="softmax"
      )

      print(f"MAPE: {mape:.6f}%")

      all_results.append((mu1, mu2, mape))

      # 暫定ベスト更新
      if mape < best_mape:
        best_mape = mape
        best_mu1 = mu1
        best_mu2 = mu2
        print("🔥 New Best!")

      print(f"Current Best → mu1={best_mu1:,}, mu2={best_mu2:,}, MAPE={best_mape:.6f}%")

    except Exception as e:
      print("⚠ Error occurred")
      print(e)

print("\n" + "=" * 60)
print("Final Best Result")
print(f"mu1 = {best_mu1:,}")
print(f"mu2 = {best_mu2:,}")
print(f"MAPE = {best_mape:.6f}%")

# 一応一覧も表示
results_df = pd.DataFrame(all_results, columns=["mu1", "mu2", "mape"])
results_df = results_df.sort_values("mape")

print("\n=== All Results (sorted) ===")
print(results_df)

# 提出用ファイル作成

In [ ]:
# ===== モデル保存（上書き） =====
joblib.dump(model, "lgbm_model.pkl")

print("モデルを保存しました: lgbm_model.pkl")

In [ ]:
import pandas as pd
import joblib
import numpy as np

# test データ読み込み
df_test = pd.read_csv("test.csv")

# 前処理
df_test_feat = make_df(df_test)

# 目的変数があれば削除
if "money_room" in df_test_feat.columns:
    df_test_feat.drop(columns=["money_room"], inplace=True)

# id列も削除
if "id" in df_test_feat.columns:
    df_test_feat.drop(columns=["id"], inplace=True)

# モデル読み込み
model = joblib.load("lgbm_model.pkl")

#通常学習
# y_pred = model.predict(df_test_feat)import pandas as pd

# log1p学習の場合
y_pred_log = model.predict(df_test_feat)
y_pred = np.expm1(y_pred_log)

# 提出用 DataFrame 作成
submit_df = pd.DataFrame({
    0: range(len(y_pred)),  # インデックス
    1: y_pred               # 予測値
})

# ヘッダなしで CSV 出力
submit_df.to_csv("submission.csv", index=False, header=False)


# メモ 内部データ確認用

メモ

4 building_status …… 1と9の二値を1と0の二値に変換

7 building_type …… 3が多くなるべきところなのに4が多すぎる。一旦1,4のみ保持してその他を0にする

12 unit_count …… NaNをbuilding_typeごとの平均値にする。1が多すぎるから木構造じゃないとうまく使えないかも

14 lon …… メッシュ化したカラムを作成

15 lat …… メッシュ化したカラムを作成

16 building_structure …… NaNを4で埋める(4……RC / 最頻の4の記入が省かれている可能性が高い為)

17 total_floor_area …… -1とNaNを中央値に。対数変換したlog_total_floor_areaカラムを追加で作成

18 building_area …… -1とNaNを中央値に。対数変換したlog_building_areaカラムを追加で作成

19 floor_count …… 0と外れ値をNaNに。bin化したfloor_count_binカラムを追加で作成

20 basement_floor_count …… 0と1の2値にする

21 year_built …… 外れ値をNaNにしてbin化したカラムの追加

22 building_land_area …… 外れ値をNaNにしてbin化した別カラムの作成

26 building_land_chimoku …… 1 or NaNでフラグ化した別カラムを作成

27 land_youto …… マッピング(1,10を合わせて低層住居専用地域に)

28 land_tosh …… マッピング

45 building_tag_id …… 使えそうなものをone-hot化。ガス,コンロを3値で保持(67unit_tag_id,145statuses と合わせて使う)

49 room_floor …… bin化。NaNはNaNとして保持

53 unit_area …… NaNを中央値で埋め、対数変換

54 floor_plan_code …… 間取りのみ取り出し → 109madori_kind_allと内容がかぶるため、一旦コメントアウト

67 unit_tag_id …… 使えそうなものをone-hot化。ガス,コンロを3値で保持(45building_tag_id,145statuses と合わせて使う)

73 bukken_type …… 1202を0、1302を1のフラグ特徴量に

79,80 addr1_1, addr1_2 …… 2つを合わせた別カラムを作成

104 flg_new …… 1をそのまま、それ以外を0に

106 room_kaisuu …… 外れ値をNaNにそれ以外をbin化

107 preprocess_snapshot_window_angle …… 東西南北でonehot化

108 madori_number_all …… bin化

109 madori_kind_all …… bin化

110 money_kyoueki …… 外れ値を処理。0円フラグ別カラムを作成。0円 or NaNフラグの別カラムを作成

123 flg_no_parking …… 4(駐車場無し)をフラグにした別カラムを作成。

131 school_ele_distance …… NaNを中央値で埋め、対数変換した別カラムを作成

134 school_jun_distance …… NaNを中央値で埋め、対数変換した別カラムを作成

136 convenience_distance …… NaNを中央値で埋め、対数変換した別カラムを作成

137 super_distance …… NaNを中央値で埋め、対数変換した別カラムを作成

145 statuses …… 使えそうなものをone-hot化。ガス,コンロを3値で保持(45building_tag_id, 67unit_tag_id と合わせて使う)

その他

create_highrise_top_floor_flag …… floor_count == room_kaisuu かつ 20 以上の最上階フラグを作成

create_station_distance …… bin化した駅までの距離カラムと対数化した駅までの距離カラムを作成する

In [ ]:
from collections import defaultdict

counts = defaultdict(int)

tag_cols = ["building_tag_id", "unit_tag_id", "statuses"]

for _, row in df[tag_cols].iterrows():
    tags_in_row = set()

    for col in tag_cols:
        val = row[col]
        if pd.isna(val):
            continue
        parts = val.split("/")
        tags_in_row.update(parts)   # setなので重複は潰れる

    # 1物件につき1回だけカウント
    for tag in tags_in_row:
        counts[tag] += 1

l = len(df)

sorted_counts = sorted(
    counts.items(),
    key=lambda x: x[1],
    reverse=True
)

for k, v in sorted_counts:
    print(k, v, v / l * 100, "%")


In [ ]:
# 全内訳を出力するコード
# print(df["floor_plan_code"].value_counts(dropna=False))

# 値が大きい順
print(df["building_land_chimoku"].value_counts(dropna=False).sort_index(ascending=False))


In [ ]:
df.nunique()

In [ ]:
df.isnull().mean() * 100

In [ ]:
import pandas as pd

def auto_column_classify(df, id_threshold=0.8, high_unique_threshold=50):
    report = []

    n_rows = len(df)

    for col in df.columns:
        nunique = df[col].nunique(dropna=True)
        dtype = df[col].dtype
        null_ratio = df[col].isnull().mean()

        # ID系（ほぼユニーク）
        if nunique == 0:
            cls = "DROP_all_missing"
        elif nunique == n_rows:
            cls = "ID_unique"
        elif nunique > n_rows * id_threshold:
            cls = "ID_like"

        # カテゴリ（ユニークが少ない or dtype=object）
        elif dtype == object and nunique <= high_unique_threshold:
            cls = "CATEGORY"
        elif nunique <= 10:
            cls = "CATEGORY"

        # 高ユニークの文字列 → 雑音
        elif dtype == object and nunique > high_unique_threshold:
            cls = "TEXT_high_unique"

        # 数値でユニーク多い → 連続値
        elif dtype != object:
            cls = "NUMERIC"

        else:
            cls = "OTHER"

        report.append({
            "column": col,
            "dtype": str(dtype),
            "nunique": nunique,
            "null_ratio": round(null_ratio * 100, 2),
            "class": cls
        })

    return pd.DataFrame(report)


# ▼ 実行
report_df = auto_column_classify(df)
report_df


# 結果推移

※Low price MAPEはmoney_roomが1000万円未満のみ対象

##【2025/12/27】

データセットのみでの学習。

(通常学習)

Validation MAPE: 34.6227

(log学習)

Validation MAPE: 30.0155

(コンペ提出結果)

Validation MAPE: 28.7300

Low price MAPE: 59.90

コンペ終了

順位 581/645

##【2026/1/14】

lon, lat, adrr1を追加。

(通常学習)

Validation MAPE: 21.9867

(log学習)

Validation MAPE: 19.7243

Low price MAPE: 35.59

順位 459/645 相当(↑122)

##【2026/2/2】

駅からの距離(OpenStreetMap)、公示価格を追加

(通常学習)

Validation MAPE: 20.5772

(log学習)

Validation MAPE: 18.5156

Low price MAPE: 33.15

順位 406/645 相当(↑53)

##【2026/2/12】

Two-Stageモデル実装(mu_1 = 15,000,000, mu_2 = 30,000,000)

(log学習)

Validation MAPE: 16.4793

Low price MAPE: 24.91

順位 272/645 相当(↑134)

# データ収集用

In [ ]:
df = make_df(df)

In [ ]:
import geopandas as gpd

gdf = gpd.read_file("L01-25.geojson")
print(gdf.head())

In [ ]:
import matplotlib.pyplot as plt

step = 5_000_000

dist = (
  (df['money_room'] // step) * step
).value_counts().sort_index()




bins = range(
  int(df['money_room'].min() // step * step),
  int(df['money_room'].max() // step * step + step),
  step
)

plt.figure()
plt.hist(df['money_room'], bins=bins, rwidth=0.9)
plt.xlabel('money_room')
plt.ylabel('count')
plt.show()




In [ ]:
df_valid = X_valid.copy()
df_valid["y_true"] = y_valid_true
df_valid["y_pred"] = y_valid_pred

# 個別MAPE
df_valid["mape"] = np.abs(
  (df_valid["y_true"] - df_valid["y_pred"]) / df_valid["y_true"]
)

df_valid = df_valid.sort_values("y_true").reset_index(drop=True)

n = len(df_valid)
window_size = int(n * 0.005)  # 0.5%
stride = int(n * 0.002)       # 0.2%（重なりあり）

records = []

for start in range(0, n - window_size + 1, stride):
  end = start + window_size
  window = df_valid.iloc[start:end]

  records.append({
    "start_idx": start,
    "end_idx": end,
    "price_min": window["y_true"].min(),
    "price_max": window["y_true"].max(),
    "price_median": window["y_true"].median(),
    "mape": window["mape"].mean()
  })

df_mape_window = pd.DataFrame(records)

df_mape_window["delta_mape"] = (
  df_mape_window["mape"].shift(1) - df_mape_window["mape"]
)

df_mape_window["mape_smooth"] = (
  df_mape_window["mape"]
  .rolling(window=3, center=True)
  .mean()
)

eps = 0.001  # 0.1%
cond = (
  df_mape_window["delta_mape"].abs() < eps
)

df_mape_window["stable"] = cond

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(df_mape_window["price_median"], df_mape_window["mape_smooth"])
plt.xlabel("price (median of window)")
plt.ylabel("MAPE")
plt.title("MAPE transition by price window")
plt.grid(True)
plt.show()


In [ ]:
df_low = df[df["money_room"] <= 15000000]
print(df_low["bukken_type"].value_counts(normalize=True))
print(df_low["addr1_1"].value_counts(normalize=True))
print(df_low["flg_new"].value_counts(normalize=True))



In [ ]:
print(df_low.groupby("addr1_1")["money_room"].std().sort_values(ascending=False))


In [ ]:
df_eval = df.copy()
df_eval["pred_soft"] = pred_soft
df_low = df_eval[df_eval["money_room"] <= 15_000_000].copy()
def calc_mape(y_true, y_pred):
  y_true = np.clip(y_true, 1, None)
  return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

pref_mape = []

for pref in df_low["addr1_1"].unique():
  df_tmp = df_low[df_low["addr1_1"] == pref]

  mape = calc_mape(
    df_tmp["money_room"].values,
    df_tmp["pred_soft"].values
  )

  pref_mape.append((pref, mape, len(df_tmp)))

pref_mape = sorted(pref_mape, key=lambda x: x[1])
pref_mape[:10]


In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold

# =========================
# データ準備
# =========================
df_train = df_t.copy(deep=True)

X = df_train.drop(columns=[TARGET])
y = df_train[TARGET].values

# =========================
# CV設定
# =========================
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# =========================
# One-stage OOF
# =========================
oof_one = np.zeros(len(X))

for tr_idx, va_idx in kf.split(X):

  model = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
  )

  model.fit(
    X.iloc[tr_idx],
    np.log1p(y[tr_idx])
  )

  pred_log = model.predict(X.iloc[va_idx])
  pred = np.expm1(pred_log)

  oof_one[va_idx] = pred

oof_one = np.clip(oof_one, 1, None)

mape_one = np.mean(np.abs((y - oof_one) / y)) * 100
print("One-stage OOF MAPE:", mape_one)


# =========================
# Two-stage (Softmax MoE)
# =========================
params = {
  "n_estimators": 1000,
  "learning_rate": 0.05,
  "num_leaves": 31,
  "random_state": 42,
  "n_jobs": -1
}

mu1 = 15_000_000
mu2 = 30_000_000

mape_soft, pred_soft = run_moe(
  X,
  y,
  kf,
  params,
  mu1=mu1,
  mu2=mu2,
  gate_type="softmax"
)

print("Two-stage Softmax OOF MAPE:", mape_soft)


# =========================
# MAPEカーブ作成関数
# =========================
def make_mape_curve(y_true, y_pred, window_ratio=0.005, min_size=30):

  df_tmp = pd.DataFrame({
    "price": y_true,
    "pred": y_pred
  })

  df_tmp["ape"] = np.abs((df_tmp["price"] - df_tmp["pred"]) / df_tmp["price"])
  df_tmp = df_tmp.sort_values("price").reset_index(drop=True)

  N = len(df_tmp)
  window_size = max(int(N * window_ratio), min_size)

  window_mape = []
  window_center_price = []

  for start in range(0, N - window_size + 1):
    end = start + window_size
    window = df_tmp.iloc[start:end]

    mape = window["ape"].mean() * 100
    center_price = window["price"].median()

    window_mape.append(mape)
    window_center_price.append(center_price)

  return pd.DataFrame({
    "price_center": window_center_price,
    "mape": window_mape
  })


# =========================
# カーブ作成
# =========================
df_curve_one  = make_mape_curve(y, oof_one)
df_curve_soft = make_mape_curve(y, pred_soft)


# =========================
# 描画
# =========================
plt.figure(figsize=(10, 5))

plt.plot(df_curve_one["price_center"],
         df_curve_one["mape"],
         label="One-stage")

plt.plot(df_curve_soft["price_center"],
         df_curve_soft["mape"],
         label="Two-stage (Softmax)")

plt.xlabel("Price")
plt.ylabel("MAPE (%)")
plt.title("MAPE Curve Comparison (0.5% window)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
def calc_mape(y_true, y_pred, mask):
  return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


mask1 = y < 15_000_000
mask2 = (y >= 15_000_000) & (y < 30_000_000)
mask3 = y >= 30_000_000


print("===== One-stage =====")
print("~1500万:", calc_mape(y, oof_one, mask1))
print("1500万~3000万:", calc_mape(y, oof_one, mask2))
print("3000万~:", calc_mape(y, oof_one, mask3))


print("\n===== Two-stage (Softmax) =====")
print("~1500万:", calc_mape(y, pred_soft, mask1))
print("1500万~3000万:", calc_mape(y, pred_soft, mask2))
print("3000万~:", calc_mape(y, pred_soft, mask3))

In [ ]:
# =========================
# 描画
# =========================
plt.figure(figsize=(10, 5))

plt.plot(df_curve_one["price_center"],
         df_curve_one["mape"],
         label="One-stage")

plt.plot(df_curve_soft["price_center"],
         df_curve_soft["mape"],
         label="Two-stage")

plt.xlabel("Price")
plt.ylabel("MAPE (%)")
plt.title("MAPE Curve Comparison (0.5% window)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(df_curve_one["price_center"],
         df_curve_one["mape"],
         label="One-stage")

plt.xlabel("Price")
plt.ylabel("MAPE (%)")
plt.title("MAPE Curve Comparison (0.5% window)")
plt.legend()
plt.grid(True)
plt.show()